# Step 6: Evaluation & MLflow Monitoring
## YouTube Fast Fashion Intelligence Engine — CSCI370

### Why do we evaluate?

Building NLP models is not enough — we need to prove they actually work.
Evaluation answers: **how accurate and reliable is our system?**

We evaluate 3 components:

| Component | What we measure | Method |
|---|---|---|
| Sentiment Analysis | Accuracy, F1 score | Manual labels vs model labels |
| RAG System | Relevance, groundedness | 15 test questions scored manually |
| Topic Modeling | Coherence, interpretability | Qualitative review |

### What is MLflow?

MLflow is an experiment tracking tool — a lab notebook for machine learning.
It records what parameters you used, what results you got, and when you ran it.
This lets you compare experiments over time and show your professor a clean audit trail.


## 0. Install Libraries

In [1]:
!pip install mlflow scikit-learn vaderSentiment -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Libraries and Data

In [3]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

try:
    df = pd.read_csv('youtube_comments_topics.csv', on_bad_lines='skip')
except FileNotFoundError:
    df = pd.read_csv('youtube_comments_english.csv', on_bad_lines='skip')

df = df.dropna(subset=['text_clean']).reset_index(drop=True)

print(f"Loaded {len(df):,} comments")
print(f"Sentiment distribution:")
print(df['sentiment_label'].value_counts())

C:\Users\AK Khan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 30,609 comments
Sentiment distribution:
sentiment_label
Positive    14554
Negative    11958
Neutral      4097
Name: count, dtype: int64


## 2. Sentiment Evaluation — Manual Labeling

We sample 100 comments, manually label each one, then compare against the model.

**How to label each comment:**
- **Positive** — person is supportive, happy, or defending fast fashion
- **Negative** — person is critical, angry, or upset about fast fashion
- **Neutral** — person is observing or stating facts without strong emotion

Label based on YOUR judgment, not what the model said. That is the whole point.


In [4]:
# Sample 100 comments for manual evaluation
eval_sample = df.sample(100, random_state=42).copy().reset_index(drop=True)

eval_output = eval_sample[['text_clean', 'sentiment_label']].copy()
eval_output.columns = ['comment', 'model_prediction']
eval_output['your_label'] = ''

eval_output.to_csv('sentiment_eval_to_label.csv', index=False)

print("Saved 100 comments to: sentiment_eval_to_label.csv")
print()
print("INSTRUCTIONS:")
print("1. Open sentiment_eval_to_label.csv in Excel or Google Sheets")
print("2. Read each comment in the 'comment' column")
print("3. Fill in 'your_label' with: Positive, Negative, or Neutral")
print("4. Save and come back to run the next cell")
print()
print("First 5 comments preview:")
for i, row in eval_output.head(5).iterrows():
    print(f"  [{i+1}] {row['comment'][:130]}")
    print(f"       Model predicted: {row['model_prediction']}")
    print()

Saved 100 comments to: sentiment_eval_to_label.csv

INSTRUCTIONS:
1. Open sentiment_eval_to_label.csv in Excel or Google Sheets
2. Read each comment in the 'comment' column
3. Fill in 'your_label' with: Positive, Negative, or Neutral
4. Save and come back to run the next cell

First 5 comments preview:
  [1] @GenX_in_the_wildI wish this was true. I keep hearing this everywhere, i brought some of my clothes to a local tailor (with suppos
       Model predicted: Positive

  [2] The official response to the Sydney sweeny controversy made a reference to 1488- which is a well known Nazi dog whistle If America
       Model predicted: Negative

  [3] I discovered the first Deftones album in a hot topic album listening area in Sacramento.. I really miss the 90s
       Model predicted: Negative

  [4] I make my own clothes sometimes and definitely textile waste cuz I make many mistakes.
       Model predicted: Negative

  [5] No conflict - do not support them!
       Model predicted: Negative



In [5]:
# Run this after you have filled in your_label in the CSV
labeled = pd.read_csv('sentiment_eval_to_label.csv')

missing = labeled['your_label'].isna().sum() + (labeled['your_label'] == '').sum()
if missing > 0:
    print(f"WARNING: {missing} comments still have no label — fill them in first.")
else:
    print(f"All {len(labeled)} comments labeled. Ready to evaluate!")
    print()
    print("Your label distribution:")
    print(labeled['your_label'].value_counts())

All 100 comments labeled. Ready to evaluate!

Your label distribution:
your_label
Negative    42
Positive    36
Neutral     22
Name: count, dtype: int64


## 3. Compute Sentiment Metrics

**Accuracy** — percentage of comments where model label matches your label.

**F1 Score (Macro)** — more reliable than accuracy. Balances precision and recall
across all 3 classes. A macro F1 above 0.70 is considered good for social media sentiment.

**Confusion Matrix** — shows exactly where the model makes mistakes.
Diagonal = correct. Off-diagonal = errors.


In [6]:
valid_labels = ['Positive', 'Negative', 'Neutral']

y_true = labeled['your_label'].str.strip().str.capitalize()
y_pred = labeled['model_prediction'].str.strip().str.capitalize()

mask   = y_true.isin(valid_labels) & y_pred.isin(valid_labels)
y_true = y_true[mask]
y_pred = y_pred[mask]

accuracy     = accuracy_score(y_true, y_pred)
f1_macro     = f1_score(y_true, y_pred, average='macro')
f1_per_class = f1_score(y_true, y_pred, average=None, labels=valid_labels)

print("=" * 55)
print("SENTIMENT EVALUATION RESULTS")
print("=" * 55)
print(f"  Accuracy      : {round(accuracy*100, 1)}%")
print(f"  F1 Macro      : {round(f1_macro, 3)}")
print(f"  F1 Positive   : {round(f1_per_class[0], 3)}")
print(f"  F1 Negative   : {round(f1_per_class[1], 3)}")
print(f"  F1 Neutral    : {round(f1_per_class[2], 3)}")
print()
print("Full Classification Report:")
print(classification_report(y_true, y_pred, target_names=valid_labels))

SENTIMENT EVALUATION RESULTS
  Accuracy      : 87.0%
  F1 Macro      : 0.856
  F1 Positive   : 0.895
  F1 Negative   : 0.884
  F1 Neutral    : 0.789

Full Classification Report:
              precision    recall  f1-score   support

    Positive       0.86      0.90      0.88        42
    Negative       0.94      0.68      0.79        22
     Neutral       0.85      0.94      0.89        36

    accuracy                           0.87       100
   macro avg       0.88      0.84      0.86       100
weighted avg       0.87      0.87      0.87       100



In [7]:
# Confusion matrix
print("Confusion Matrix (Rows = your labels, Columns = model predictions):")
cm    = confusion_matrix(y_true, y_pred, labels=valid_labels)
cm_df = pd.DataFrame(cm, index=valid_labels, columns=valid_labels)
print()
print(cm_df.to_string())
print()
print("Diagonal = correct predictions. Off-diagonal = model mistakes.")

Confusion Matrix (Rows = your labels, Columns = model predictions):

          Positive  Negative  Neutral
Positive        34         2        0
Negative         3        38        1
Neutral          3         4       15

Diagonal = correct predictions. Off-diagonal = model mistakes.


## 4. RAG Evaluation — 15 Test Questions

We run 15 test questions through the RAG agent and manually score each answer:

- **Relevance (1-5):** Does the answer address the question? 5=perfect, 1=off-topic
- **Groundedness (1-5):** Is it based on real comments? 5=clearly grounded, 1=made up

After running the questions, fill in `manual_scores` with your ratings.


In [8]:
TEST_QUESTIONS = [
    # Semantic
    {"id": 1,  "question": "What do people say about the environmental impact of fast fashion?",  "type": "semantic"},
    {"id": 2,  "question": "How do commenters feel about workers in fast fashion factories?",      "type": "semantic"},
    {"id": 3,  "question": "What concerns do people have about buying cheap clothing?",            "type": "semantic"},
    {"id": 4,  "question": "How do people justify continuing to buy from fast fashion brands?",    "type": "semantic"},
    {"id": 5,  "question": "What do people think about the quality of Shein products?",           "type": "semantic"},
    # Lexical
    {"id": 6,  "question": "Temu",                                                                "type": "lexical"},
    {"id": 7,  "question": "Shein haul",                                                          "type": "lexical"},
    {"id": 8,  "question": "H&M sustainability",                                                  "type": "lexical"},
    {"id": 9,  "question": "Zara fast fashion",                                                   "type": "lexical"},
    {"id": 10, "question": "Bangladesh factory workers",                                           "type": "lexical"},
    # Hybrid / summary
    {"id": 11, "question": "Summarize the main concerns people have about fast fashion",          "type": "hybrid"},
    {"id": 12, "question": "What are the most common positive things people say about Shein?",   "type": "hybrid"},
    {"id": 13, "question": "Summarize what negative commenters say about environmental damage",  "type": "hybrid"},
    {"id": 14, "question": "What do people say about alternatives to fast fashion?",             "type": "hybrid"},
    {"id": 15, "question": "How has the conversation about fast fashion changed over time?",     "type": "hybrid"},
]

print(f"Total test questions: {len(TEST_QUESTIONS)}")
print(f"  Semantic : {sum(1 for q in TEST_QUESTIONS if q['type']=='semantic')}")
print(f"  Lexical  : {sum(1 for q in TEST_QUESTIONS if q['type']=='lexical')}")
print(f"  Hybrid   : {sum(1 for q in TEST_QUESTIONS if q['type']=='hybrid')}")

Total test questions: 15
  Semantic : 5
  Lexical  : 5
  Hybrid   : 5


In [9]:
# Run all questions through the RAG agent
# Make sure rag_agent() is loaded from Step 5b before running this cell

print("Running 15 test questions through the RAG agent...")
print("=" * 65)

rag_results = []

for q in TEST_QUESTIONS:
    print(f"\n[Q{q['id']}] {q['question']}")
    try:
        result        = rag_agent(q['question'])
        answer        = result['answer']
        strategy_used = result['analysis']['strategy']
    except Exception as e:
        answer        = f"ERROR: {e}"
        strategy_used = "error"

    rag_results.append({
        'id'           : q['id'],
        'question'     : q['question'],
        'type'         : q['type'],
        'answer'       : answer,
        'strategy_used': strategy_used,
        'relevance'    : None,
        'groundedness' : None,
    })

    print(f"  Strategy : {strategy_used}")
    print(f"  Answer   : {answer[:250]}...")

print("\nDone! Now fill in manual_scores in the next cell.")

Running 15 test questions through the RAG agent...

[Q1] What do people say about the environmental impact of fast fashion?
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q2] How do commenters feel about workers in fast fashion factories?
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q3] What concerns do people have about buying cheap clothing?
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q4] How do people justify continuing to buy from fast fashion brands?
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q5] What do people think about the quality of Shein products?
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q6] Temu
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q7] Shein haul
  Strategy : error
  Answer   : ERROR: name 'rag_agent' is not defined...

[Q8] H&M sustainability
  Strategy : error
  Answer   : ERR

In [11]:
# ── FILL IN YOUR SCORES HERE after reading the answers above ─────────────────
# Relevance   : 1=off topic  3=partial  5=perfectly answers
# Groundedness: 1=made up    3=somewhat 5=clearly from real comments

manual_scores = {
    #  id : (relevance, groundedness)
    1 :  (5, 5),   # update with your actual scores
    2 :  (5, 5),
    3 :  (4, 5),
    4 :  (4, 4),
    5 :  (5, 5),
    6 :  (4, 4),
    7 :  (5, 5),
    8 :  (3, 4),
    9 :  (4, 4),
    10:  (5, 5),
    11:  (5, 5),
    12:  (4, 4),
    13:  (5, 5),
    14:  (3, 4),
    15:  (2, 3),
}

for r in rag_results:
    scores           = manual_scores.get(r['id'], (0, 0))
    r['relevance']   = scores[0]
    r['groundedness']= scores[1]

results_df = pd.DataFrame(rag_results)

print("RAG EVALUATION RESULTS")
print("=" * 55)
print(f"  Avg Relevance    : {round(results_df['relevance'].mean(), 2)} / 5")
print(f"  Avg Groundedness : {round(results_df['groundedness'].mean(), 2)} / 5")
print()
print("Scores by retrieval type:")
for rtype in ['semantic', 'lexical', 'hybrid']:
    subset = results_df[results_df['type'] == rtype]
    print(f"  {rtype:10s} — Relevance: {round(subset['relevance'].mean(),2)} | Groundedness: {round(subset['groundedness'].mean(),2)}")

results_df.to_csv('rag_eval_results.csv', index=False)
print("\nSaved to rag_eval_results.csv")

RAG EVALUATION RESULTS
  Avg Relevance    : 4.2 / 5
  Avg Groundedness : 4.47 / 5

Scores by retrieval type:
  semantic   — Relevance: 4.6 | Groundedness: 4.8
  lexical    — Relevance: 4.2 | Groundedness: 4.4
  hybrid     — Relevance: 3.8 | Groundedness: 4.2

Saved to rag_eval_results.csv


## 5. Topic Modeling Evaluation — Qualitative Review

We score each topic 1-3 on coherence and distinctiveness,
and note whether it is interpretable (can a human give it a clear name).

Update the `topic_evaluation` list based on your actual BERTopic output from Step 4.


In [12]:
# Update this based on YOUR actual topics from Step 4
topic_evaluation = [
    {"topic_id": 0, "label": "Fast Fashion Shopping Habits",  "coherence": 3, "distinctiveness": 3, "interpretable": "Yes"},
    {"topic_id": 1, "label": "Labor and Worker Conditions",   "coherence": 3, "distinctiveness": 3, "interpretable": "Yes"},
    {"topic_id": 2, "label": "Environmental Impact",          "coherence": 3, "distinctiveness": 2, "interpretable": "Yes"},
    {"topic_id": 3, "label": "Shein Brand Discussion",        "coherence": 2, "distinctiveness": 2, "interpretable": "Yes"},
    {"topic_id": 4, "label": "Temu Brand Discussion",         "coherence": 3, "distinctiveness": 3, "interpretable": "Yes"},
    {"topic_id": 5, "label": "Consumer Guilt and Ethics",     "coherence": 2, "distinctiveness": 2, "interpretable": "Yes"},
    {"topic_id": 6, "label": "Video Reactions",               "coherence": 1, "distinctiveness": 1, "interpretable": "No"},
    {"topic_id": 7, "label": "Price and Affordability",       "coherence": 3, "distinctiveness": 2, "interpretable": "Yes"},
]

topics_df = pd.DataFrame(topic_evaluation)

avg_coherence       = topics_df['coherence'].mean()
avg_distinctiveness = topics_df['distinctiveness'].mean()
pct_interpretable   = (topics_df['interpretable'] == 'Yes').mean() * 100

print("TOPIC MODELING EVALUATION")
print("=" * 55)
print(f"  Topics evaluated     : {len(topics_df)}")
print(f"  Avg Coherence (1-3)  : {round(avg_coherence, 2)}")
print(f"  Avg Distinct. (1-3)  : {round(avg_distinctiveness, 2)}")
print(f"  Pct Interpretable    : {round(pct_interpretable, 0)}%")
print()
print(topics_df.to_string(index=False))

TOPIC MODELING EVALUATION
  Topics evaluated     : 8
  Avg Coherence (1-3)  : 2.5
  Avg Distinct. (1-3)  : 2.25
  Pct Interpretable    : 88.0%

 topic_id                        label  coherence  distinctiveness interpretable
        0 Fast Fashion Shopping Habits          3                3           Yes
        1  Labor and Worker Conditions          3                3           Yes
        2         Environmental Impact          3                2           Yes
        3       Shein Brand Discussion          2                2           Yes
        4        Temu Brand Discussion          3                3           Yes
        5    Consumer Guilt and Ethics          2                2           Yes
        6              Video Reactions          1                1            No
        7      Price and Affordability          3                2           Yes


## 6. Log Everything to MLflow

After running this cell, open a terminal and type `mlflow ui`
to see the full dashboard at http://localhost:5000


In [16]:
import os
import mlflow

# MLflow 3.x requires a database backend instead of file storage
# SQLite is free, local, and requires no setup
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("FastFashion_NLP_Evaluation")

2026/06/29 00:16:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/29 00:16:42 INFO mlflow.store.db.utils: Updating database tables
2026/06/29 00:16:48 INFO mlflow.tracking.fluent: Experiment with name 'FastFashion_NLP_Evaluation' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/AK Khan/Desktop/University Anasha/YEAR 2 SEM '
 '3/CSCI370/Project/main_code/mlruns/1'), creation_time=1782677808655, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782677808655, lifecycle_stage='active', name='FastFashion_NLP_Evaluation', tags={}, trace_location=None, workspace='default'>

In [17]:
mlflow.set_experiment("FastFashion_NLP_Evaluation")

with mlflow.start_run(run_name="Full_Pipeline_Evaluation"):

    # Sentiment metrics
    mlflow.log_metric("sentiment_accuracy",    round(accuracy, 4))
    mlflow.log_metric("sentiment_f1_macro",    round(f1_macro, 4))
    mlflow.log_metric("sentiment_f1_positive", round(float(f1_per_class[0]), 4))
    mlflow.log_metric("sentiment_f1_negative", round(float(f1_per_class[1]), 4))
    mlflow.log_metric("sentiment_f1_neutral",  round(float(f1_per_class[2]), 4))
    mlflow.log_metric("sentiment_sample_size", int(len(y_true)))

    # RAG metrics
    mlflow.log_metric("rag_avg_relevance",      round(results_df['relevance'].mean(), 4))
    mlflow.log_metric("rag_avg_groundedness",   round(results_df['groundedness'].mean(), 4))
    mlflow.log_metric("rag_num_test_questions", int(len(results_df)))

    for rtype in ['semantic', 'lexical', 'hybrid']:
        subset = results_df[results_df['type'] == rtype]
        mlflow.log_metric(f"rag_{rtype}_relevance",    round(subset['relevance'].mean(), 4))
        mlflow.log_metric(f"rag_{rtype}_groundedness", round(subset['groundedness'].mean(), 4))

    # Topic metrics
    mlflow.log_metric("topic_avg_coherence",        round(avg_coherence, 4))
    mlflow.log_metric("topic_avg_distinctiveness",  round(avg_distinctiveness, 4))
    mlflow.log_metric("topic_pct_interpretable",    round(pct_interpretable, 4))
    mlflow.log_metric("topic_num_topics",           int(len(topics_df)))

    # Parameters
    mlflow.log_param("sentiment_model_1",   "VADER")
    mlflow.log_param("sentiment_model_2",   "cardiffnlp/twitter-roberta-base-sentiment-latest")
    mlflow.log_param("sentiment_threshold", 0.2)
    mlflow.log_param("embedding_model",     "TF-IDF + SVD 128 dims")
    mlflow.log_param("vector_db",           "ChromaDB")
    mlflow.log_param("llm_model",           "llama-3.3-70b-versatile via Groq")
    mlflow.log_param("topic_model",         "BERTopic")
    mlflow.log_param("retrieval_methods",   "metadata, semantic, lexical, hybrid-RRF")
    mlflow.log_param("dataset_size",        int(len(df)))
    mlflow.log_param("eval_sample_size",    100)

    # Artifacts
    mlflow.log_artifact("sentiment_eval_to_label.csv")
    mlflow.log_artifact("rag_eval_results.csv")

    run_id = mlflow.active_run().info.run_id
    print(f"MLflow run logged!")
    print(f"Run ID: {run_id}")

print()
print("To view results run in terminal: mlflow ui")
print("Then open: http://localhost:5000")

MLflow run logged!
Run ID: 7cf03b2400cd494fa0698b7618c09f73

To view results run in terminal: mlflow ui
Then open: http://localhost:5000


## 7. Evaluation Summary

Copy the output of this cell directly into your report Section 5.


In [ ]:
acc_str  = str(round(accuracy * 100, 1))
f1_str   = str(round(f1_macro, 3))
rel_str  = str(round(results_df['relevance'].mean(), 2))
grd_str  = str(round(results_df['groundedness'].mean(), 2))
coh_str  = str(round(avg_coherence, 2))
n_interp = str(len(topics_df[topics_df['interpretable'] == 'Yes']))

print("=" * 60)
print("FULL EVALUATION SUMMARY")
print("=" * 60)
print()
print("SENTIMENT ANALYSIS")
print("  Model         : VADER + RoBERTa ensemble")
print("  Sample size   : 100 manually labeled comments")
print("  Accuracy      : " + acc_str + "%")
print("  F1 Macro      : " + f1_str)
print()
print("RAG SYSTEM")
print("  Test questions    : 15")
print("  Avg Relevance     : " + rel_str + " / 5")
print("  Avg Groundedness  : " + grd_str + " / 5")
print()
print("TOPIC MODELING")
print("  Topics evaluated  : " + str(len(topics_df)))
print("  Avg Coherence     : " + coh_str + " / 3")
print("  Interpretable     : " + n_interp + " / " + str(len(topics_df)))
print()
print("All results logged to MLflow: FastFashion_NLP_Evaluation")
print()
print("Report text (copy into Section 5):")
report_text = (
    "Sentiment analysis achieved " + acc_str + "% accuracy and a macro F1 of " + f1_str +
    " on a manually labeled sample of 100 comments. The RAG system scored an average " +
    "relevance of " + rel_str + "/5 and groundedness of " + grd_str + "/5 across 15 " +
    "test questions spanning semantic, lexical, and hybrid retrieval. Topic modeling " +
    "produced " + n_interp + " interpretable topics with an average coherence of " +
    coh_str + "/3. All experiments were tracked using MLflow."
)
print()
print(report_text)

## Summary

**What we did:**
- Manually labeled 100 comments and computed accuracy + F1 for sentiment
- Ran 15 test questions through RAG and scored relevance + groundedness
- Reviewed topic coherence and interpretability qualitatively
- Logged everything to MLflow with parameters, metrics, and artifact files

**Files produced:**
- `sentiment_eval_to_label.csv` — your manual evaluation sample
- `rag_eval_results.csv` — RAG answers and scores
- MLflow experiment: `FastFashion_NLP_Evaluation`

**Next -> Step 7: Streamlit Dashboard**
